# QC filtering 01 — Removed fraction by genotype

Does Pal et al.'s per-sample QC remove cells at different rates in BRCA1+/- donors
and premenopausal WT donors? The donor is the unit of replication. This series
answers that in two phases. Phase 1 does not transfer labels.

| Notebook | Question |
|---|---|
| `00` cohort | Which 12 donors are in the comparison, and which analysis choices are frozen before any test? |
| `01` filtering rate | Does the removed fraction, overall and by reason, differ by genotype? |
| `02` Pal reference | Do the authors' own labels reproduce their retained-cell result (P = 0.14)? |
| `03` tipping point | How LP-enriched would BRCA1 removed cells have to be to hide an LP expansion? |

**This notebook tests the removed fraction.** The primary test is the overall
removed fraction, two-sided, at alpha 0.01. It is not part of the reason family.
The four non-exclusive reasons are secondary, two-sided, at alpha 0.02, with a
Benjamini-Hochberg adjustment across those four. Exclusive reasons, and cells that
fail several filters, are described and are not tested.

The uniform, MAD, and depth-equalized re-runs are not extra hypotheses. Uniform and
depth-equalized re-runs are the pre-specified stability check. The MAD re-run is
reported and left out of that verdict. Thresholds, the depth target, and the seeds
come from notebook 00. A non-significant result is inconclusive. The minimum
detectable effect is reported beside the estimate.

Menopause is not stratified. Two BRCA1 donors are post-oophorectomy and every WT
donor is premenopausal, so a genotype contrast is also a menopause contrast.

In [ ]:
import logging

import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

from signals_in_the_noise.analysis.filtering_rate import (
    load_phase1,
    load_specimen_adatas,
    run_filtering_rate,
    write_filtering_rate_outputs,
)
from signals_in_the_noise.analysis.qc_filtering import output_directory
from signals_in_the_noise.config import get_data_path
from signals_in_the_noise.utils.logging_config import setup_logging

setup_logging(logging.INFO)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

OUTPUT_DIR = output_directory()
OUTPUT_DIR

## 1. Frozen cohort

Cell and donor tables, and the Phase 1 parameters, were written by notebook 00.
Only the 12 cohort cache files are loaded. The uniform rule is the frozen modal
thresholds, not a mode recomputed here. After depth equalization the same frozen
rule is applied to the thinned counts.

In [ ]:
phase1 = load_phase1(OUTPUT_DIR)
donors = pd.read_csv(OUTPUT_DIR / "donors.csv", index_col=0)
cells = pd.read_csv(OUTPUT_DIR / "cells.csv")
adatas = load_specimen_adatas(donors.index, get_data_path("GSE161529_adata_cache"))
len(adatas), cells["donor"].nunique()

## 2. Tests, stability check, and handoff

The strip plot marks source with the point shape and menopause with the color.
The handoff below is the text to paste back after this notebook runs.

In [ ]:
result = run_filtering_rate(cells, donors, adatas, phase1, rng=np.random.default_rng(0))
handoff = write_filtering_rate_outputs(result, cells, donors, phase1, OUTPUT_DIR)
display(result["primary"])
display(result["secondary"])
display(result["robustness"])
display(result["spearman"])
display(Image(filename=str(OUTPUT_DIR / "01-removal-fractions.png")))
display(Markdown(handoff))